In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

df = pd.read_csv('solar_system.csv')

#a)
print("Before:", df.shape)
df = df.set_index("Attribute").T
print("After:", df.shape)

df.index.name = "Planet"
df.reset_index(inplace=True)
df.columns.name = None

print(df)

#2.1.1 --> question 5

#b)
#Before: rows = 5 and columns = 10
#After: rows = 10 and columns = 5

#c)
print("all of the imports allow for the data handling, plotting, and scientific constants/units")
print("df.pd.read --> reads the CSV file into a dataframe")
print("print(df.head()) --> displays the first five rows of the dataframe")
print("df = df.set_index('Attribute').T --> sets 'attribute' as the row index and the .T transposes the dataframe")
print("df.index.name = 'Planet' --> renames the row index label to 'planet'")
print("df.reset_index(inplace=True)--> converts the index into a regular column")
print("df.columns.name = None --> removes the column axis name for cleaner formatting")

#d) 
print("because you're not adding data, just reorganizing it")

#e)
print(df.columns)

#f)
print("columns with units(5): mass, diameter, density, gravity, escape velocity AND columns without units (1): planet")

#g)
print(df)

#2.1.2

print(df.dtypes)

for col in df.columns:
    print(col)
    print(df[col].apply(type).value_counts())
    print()

#a) 
print("they are actually strings")
#b)
print("it will either fail or give incorrect results")

#2.1.3
for col in df.columns:
    if col not in ["Planet", "Ring System?", "Global Magnetic Field?"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)

#a)
print("the three diff data types now seen are float64, object, and bool")

#b)
print("float64 means a numeric data type used for continuous values")
print("object means a general catch-all type in pandas and stores strings")
print("bool stores true/false values")

#c)
print("the columns that'll still be objects are planet, ring system, and global magnetic field because they're not numeric")

#2.2
def attach_units(col_name, unit, new_name=None):
    new_values = []
    for val in df[col_name]:
        new_values.append(val * unit)
    df[col_name] = new_values
    if new_name:
        df.rename(columns={col_name: new_name}, inplace=True)

attach_units("Mass (10^24kg)", 1e24 * u.kg, "Mass (kg)")
print(df["Mass (kg)"])

def attach_units(col_name, unit, new_name=None):
    df[col_name] = [val * unit for val in df[col_name]]
    if new_name:
        df.rename(columns={col_name: new_name}, inplace=True)

#2.3
peri_col = "Perihelion (10^6 km)"
aph_col = "Aphelion (10^6 km)"

semi_major_axis = ((df[peri_col] + df[aph_col]) / 2) * 1e6 * u.km

aph_index = df.columns.get_loc(aph_col)
df.insert(aph_index + 1, "Semi-Major Axis (km)", semi_major_axis)

print(df.columns)
print(df["Semi-Major Axis (km)"])

old_col = "Orbital Period (days)"
new_col = "Orbital Period (years)"

df[old_col] = df[old_col].apply(lambda x: (x * u.day).to(u.year)) 
df.rename(columns={old_col: new_col}, inplace=True)

print(df.columns)
print(df[new_col])

favorite="Jupiter"
period=df.loc[df["Planet"] == favorite, new_col].iloc[0]

print(f"{favorite}'s orbital period is {period.value:.4f} {period.unit}")

print("AU means Astronmical Unit. It's the average distance between Earth and the Sun.")
print("One AU using astropy:")
print(const.au.to(u.km))

distance_cols = [
    "Perihelion (10^6 km)",
    "Aphelion (10^6 km)",
    "Semi-Major Axis (km)"
]
for col in distance_cols:
    if col in df.columns:
        if "10^6" in col:
            values_km = df[col].values * 1e6 * u.km
        else:
            values_km = df[col].values * u.km
        new_name = col.replace("(10^6 km)", "(AU)").replace("(km)", "(AU)")
        df[col] = values_km.to(u.AU)
        df.rename(columns={col: new_name}, inplace=True)
print(df.columns)
favorite = "Jupiter"

for col in df.columns:
    if "(AU)" in col:
        value = df.loc[df["Planet"] == favorite, col].iloc[0]
        print(f"{favorite} {col}: {value}")

for col in df.columns:
    df[col] = df[col].apply(lambda x: str(x) if pd.notnull(x) else "")
    
df.to_csv('units.csv', index=False)

Before: (20, 11)
After: (10, 20)
    Planet Mass (10^24kg) Diameter (km) Density (kg/m^3) Gravity (m/s^2)  \
0  Mercury          0.330          4879             5429             3.7   
1    Venus           4.87         12104             5243             8.9   
2    Earth           5.97         12756             5514             9.8   
3     Moon          0.073          3475             3340             1.6   
4     Mars          0.642          6792             3934             3.7   
5  Jupiter           1898        142984             1326            23.1   
6   Saturn            568        120536              687             9.0   
7   Uranus           86.8         51118             1270             8.7   
8  Neptune            102         49528             1638            11.0   
9    Pluto         0.0130          2376             1850             0.7   

  Escape Velocity (km/s) Rotation Period (hours) Length of Day (hours)  \
0                    4.3                  1407.6        